<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# PySpark Fundamentals: DataFrames (Databricks)

*Session 5 · Notebook 04.02 · Lecture · Coach version · Databricks*

## Overview

Everything so far has run in **pandas and scikit-learn on a single machine**. When a dataset is too big to fit in one computer's memory, we need **distributed** computing, and **Apache Spark** is the standard tool. **PySpark** is its Python interface.

This notebook teaches the Spark **DataFrame** API, which will feel familiar after pandas: starting a Spark session, creating and reading DataFrames, inspecting and selecting data, adding columns, filtering, handling missing values, grouping and aggregating, querying with SQL, and bridging back to pandas for plotting. We use one realistic dataset throughout, **Titanic**, the same data you use in the PySpark lab (04.04) and the MLlib classification lecture (04.05). Notebook 04.03 then builds machine-learning models on top of these skills with Spark MLlib.

## Learning Objectives

By the end of this notebook you will be able to:

- Explain what Spark and PySpark are, and when to reach for them instead of pandas.
- Start a `SparkSession`, define a schema, and create or read Spark DataFrames.
- Inspect data and understand transformations versus actions (`show`, `collect`, `take`).
- Select and create columns, filter rows, and handle missing values.
- Group and aggregate, and bring results into pandas with `toPandas()` to plot them.

## Prerequisites

- The pandas work in Sessions 2 and 3 (the ideas map directly onto Spark DataFrames).
- A **Spark environment** (see the setup note below). The easiest is Google Colab.

## Running this notebook on Databricks

This is the **Databricks** version. On Databricks you do not install Spark or create a session yourself:

- A **SparkSession is already provided** as `spark` (the `getOrCreate()` calls below simply return it).
- **No `pip install pyspark`** is needed; the cluster provides Spark.
- **Upload the data once to DBFS**, for example to `/FileStore/cbs_datasets/Session_5/`, via *Catalog / DBFS / Upload* (or a Unity Catalog **Volume**), then point `DATA_PATH` below at it. Spark reads it with the `dbfs:/` scheme.
- Do **not** call `spark.stop()`; the cluster manages the session (the stop cells below are commented out).
- Tip: you can use Databricks' built-in `display(df)` instead of `df.show()` for richer tables and charts.

In [ ]:
# Where the data lives. Locally this points at the repo datasets folder;
# in Colab, upload train_titanic.csv and set DATA_PATH = "".
DATA_PATH = "dbfs:/FileStore/cbs_datasets/Session_5/"   # DBFS location where you uploaded the data

## Index

1. [Why this matters for risk analysis](#sec1)
2. [What is PySpark?](#sec2)
3. [Starting a SparkSession](#sec3)
4. [Creating a DataFrame and defining a schema](#sec4)
5. [Reading data](#sec5)
6. [Inspecting data: show, collect and take](#sec6)
7. [Selecting and creating columns](#sec7)
8. [Filtering rows](#sec8)
9. [Handling missing data](#sec9)
10. [Grouping, aggregating and plotting](#sec10)
11. [Querying with Spark SQL](#sec11)
12. [Writing data out](#sec12)
13. [Key Takeaways](#takeaways)

<a id="sec1"></a>
# Section 1: Why this matters for risk analysis

Risk data can be enormous: every card transaction, every account event, years of market ticks. When a table no longer fits in one machine's memory, pandas stops working and you need a tool that spreads the data and the computation across many machines.

| Situation | Why Spark helps |
|---|---|
| Data larger than memory | Spark partitions the data across a cluster and processes it in parallel |
| Slow single-machine jobs | Work is distributed, so large aggregations and joins run far faster |
| Production data pipelines | Spark is the standard engine behind large data platforms (Databricks, EMR) |
| Same skills, bigger data | The DataFrame API mirrors pandas, so what you know transfers |

The concepts you already know (select, filter, group by, handle missing values) all exist in Spark, just with slightly different syntax and one important twist about **when** the work actually happens.

<a id="sec2"></a>
# Section 2: What is PySpark?

**Definition:** **Apache Spark** is an engine for processing large datasets across a cluster of machines in parallel. **PySpark** is its Python API. You write Python; Spark distributes the work.

**Example:** averaging a column over a billion rows: pandas would run out of memory, while Spark splits the rows across the cluster, averages each part, and combines the results.

**Analogy:** pandas is one chef cooking a meal alone; Spark is a head chef coordinating a whole kitchen brigade, each cook handling part of the order at the same time.

**Explanation:**

- **Lazy evaluation is the key twist.** Spark separates **transformations** (define *what* to do, for example `select` or `filter`) from **actions** (trigger the actual computation, for example `show` or `collect`). Transformations are queued and only run when an action is called, which lets Spark optimise the whole plan before executing it.
- Spark has a low-level API (**RDDs**) and a high-level one (**DataFrames**). Modern PySpark work is almost entirely DataFrames (they are faster and far easier), so this notebook focuses on DataFrames throughout.

<a id="sec3"></a>
# Section 3: Starting a SparkSession

Every PySpark program starts by creating a `SparkSession`, the entry point to Spark. `getOrCreate()` reuses one if it already exists.

In [ ]:
# On Databricks the cluster already provides the `spark` session (no import or creation needed).
# Local / non-Databricks environments would instead need:
#   from pyspark.sql import SparkSession
#   spark = SparkSession.builder.appName("PySparkFundamentals").getOrCreate()

<a id="sec4"></a>
# Section 4: Creating a DataFrame and defining a schema

A Spark **DataFrame** is a distributed table with named, typed columns, much like a pandas DataFrame. You can build one from Python data, and this is the natural place to meet a **schema** (an explicit list of column names and types). Defining a schema is essential when a file has no header row, which is exactly the case for the `cal_housing.data` file used in notebook 24.

In [ ]:
import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# A schema states each column's name and type
schema = StructType([
    StructField("name", StringType(), True),      # True = nullable
    StructField("dept", StringType(), True),
    StructField("salary", IntegerType(), True),
])

data = [("Ada", "Risk", 62000),
        ("Grace", "Data", 71000),
        ("Alan", "Risk", 58000)]

df_people = spark.createDataFrame(data=data, schema=schema)   # build a DataFrame from Python data
df_people.show()
df_people.printSchema()

<a id="sec5"></a>
# Section 5: Reading data

Most of the time you read data from files. Spark reads CSV, JSON, Parquet and more. We load the Titanic training data and let Spark infer the types from the header row. (For large or headerless files you would pass an explicit `schema` instead, as in Section 4.)

In [ ]:
# header=True uses the first row as column names; inferSchema reads types automatically.
URL = "https://rockborne-bucket-01-cbs.s3.eu-west-2.amazonaws.com/Data_Sources_CBS_Risk/Session_5/"
# Scenario A (active): read from the public S3 bucket over HTTPS with pandas, then hand it to Spark.
pdf = pd.read_csv(URL + "train_titanic.csv")
# Titanic has missing values: turn NaN in the TEXT columns into None so Spark infers types cleanly.
for _c in pdf.columns:
    if pdf[_c].dtype == object:
        pdf[_c] = pdf[_c].where(pd.notnull(pdf[_c]), None)
titanic = spark.createDataFrame(pdf)

# Other ways to load the same data (uncomment the scenario you need):
# Scenario B - Spark reads directly from S3 (s3:// works on Databricks):
# titanic = spark.read.csv("s3://rockborne-bucket-01-cbs/Data_Sources_CBS_Risk/Session_5/train_titanic.csv", header=True, inferSchema=True)
# Scenario C - pandas over HTTPS via a config file, then to Spark:
# from config import session_datasets_http
# titanic = spark.createDataFrame(pd.read_csv(session_datasets_http["train_titanic"]))
# Scenario D - Spark reads s3:// via a config file:
# from config import session_datasets
# titanic = spark.read.csv(session_datasets["train_titanic"], header=True, inferSchema=True)
# Scenario E - a file you uploaded to DBFS:
# titanic = spark.read.csv("dbfs:/FileStore/cbs_datasets/Session_5/train_titanic.csv", header=True, inferSchema=True)

titanic.printSchema()
titanic.show(5)

<a id="sec6"></a>
# Section 6: Inspecting data: show, collect and take

These actions all return data, but differently, and the distinction matters at scale.

- `show(n)`: prints the first `n` rows as a formatted table (for humans).
- `take(n)`: returns the first `n` rows as a Python list of `Row` objects (for code).
- `collect()`: returns **all** rows to the driver as a list. Safe on small data, but **dangerous on big data** (it can overwhelm the driver's memory).

In [ ]:
print("Columns:", titanic.columns)          # column names
print("Row count:", titanic.count())          # an action: counts across the cluster
titanic.describe(["Age", "Fare"]).show()      # summary statistics for chosen columns

print(titanic.take(2))                        # first 2 rows as Row objects
# Avoid titanic.collect() on large data: it pulls every row to one machine.

<a id="sec7"></a>
# Section 7: Selecting and creating columns

Selecting and adding columns are **transformations**: they return a new DataFrame (Spark DataFrames are immutable) and do nothing until an action runs.

In [ ]:
titanic.select("Survived", "Pclass", "Sex", "Age").show(5)     # pick columns

# Create a new feature: family size = siblings/spouses + parents/children + the passenger
titanic = titanic.withColumn("FamilySize", titanic["SibSp"] + titanic["Parch"] + 1)
titanic.select("Name", "SibSp", "Parch", "FamilySize").show(5)

# Rename a column (returns a new DataFrame)
titanic.withColumnRenamed("Pclass", "PassengerClass").show(3)

Note that `withColumn` returns a **new** DataFrame. We reassigned `titanic = titanic.withColumn(...)` above to keep the new `FamilySize` column; without the assignment the change would be discarded.

<a id="sec8"></a>
# Section 8: Filtering rows

You can filter with a SQL-style string or with column expressions. With column expressions, combine conditions using `&` (and), `|` (or) and `~` (not), wrapping each condition in parentheses.

In [ ]:
# SQL-style string
titanic.filter("Survived == 1 AND Pclass == 1").select("Name", "Age", "Fare").show(5)

# Column-expression style (note the parentheses around each condition)
titanic.filter((titanic["Age"] < 18) & (titanic["Survived"] == 1)).select("Name", "Age").show(5)

# Count how many children (under 18) were on board
print("Children under 18:", titanic.filter(titanic["Age"] < 18).count())

<a id="sec9"></a>
# Section 9: Handling missing data

Real data has gaps, and Titanic is a good example: `Age`, `Cabin` and `Embarked` all have missing values. Spark handles nulls through the `.na` interface (`drop` and `fill`), and you can count nulls per column with a small expression.

In [ ]:
from pyspark.sql.functions import col, count, when

# Count nulls in every column
titanic.select([count(when(col(c).isNull(), c)).alias(c) for c in titanic.columns]).show()

In [ ]:
# Dropping is one option, but it can throw away a lot of data (Age alone is missing ~20%)
print("Rows before dropping:", titanic.count())
print("Rows if we drop any null:", titanic.na.drop().count())

# Usually we FILL instead. Sensible choices per column:
from pyspark.sql.functions import mean

age_mean = titanic.select(mean("Age")).collect()[0][0]        # mean age across the cluster
titanic = titanic.na.fill({
    "Age": age_mean,          # numeric: fill with the mean
    "Embarked": "S",          # categorical: fill with the most common port
    "Cabin": "Unknown",       # text: a placeholder
})
# Confirm there are no more nulls in these columns
titanic.select([count(when(col(c).isNull(), c)).alias(c) for c in ["Age", "Embarked", "Cabin"]]).show()

<a id="sec10"></a>
# Section 10: Grouping, aggregating and plotting

`groupBy` plus an aggregate mirrors pandas. Crucially, to **plot** Spark results you first bring the (small) aggregated table into pandas with `toPandas()`, then use matplotlib. This "aggregate in Spark, plot in pandas" pattern is used throughout the MLlib notebooks, so it is worth getting comfortable with now.

In [ ]:
# Survival rate by passenger class, computed in Spark
by_class = titanic.groupBy("Pclass").mean("Survived").orderBy("Pclass")
by_class.show()

# Other common aggregates
titanic.groupBy("Sex").count().show()                 # rows per group
titanic.agg({"Fare": "max"}).show()                    # a single aggregate over the whole frame

In [ ]:
import matplotlib.pyplot as plt

# Bring the SMALL aggregated result into pandas, then plot it.
# (toPandas() pulls data to the driver, so only ever call it on aggregated or sampled data.)
surv_by_class = titanic.groupBy("Pclass").mean("Survived").orderBy("Pclass").toPandas()

plt.figure(figsize=(6, 4))
plt.bar(surv_by_class["Pclass"].astype(str), surv_by_class["avg(Survived)"], color="steelblue")
plt.xlabel("passenger class"); plt.ylabel("survival rate")
plt.title("Survival rate by class (aggregated in Spark, plotted in pandas)")
plt.show()

<a id="sec11"></a>
# Section 11: Querying with Spark SQL

If you know SQL, you can query a DataFrame directly: register it as a temporary view, then run `spark.sql(...)`.

In [ ]:
titanic.createOrReplaceTempView("titanic")          # register the view once

spark.sql("""
    SELECT Pclass, ROUND(AVG(Fare), 2) AS avg_fare
    FROM titanic
    GROUP BY Pclass
    ORDER BY avg_fare DESC
""").show()

<a id="sec12"></a>
# Section 12: Writing data out

Spark writes to a **folder** of part-files (one per partition), which is how distributed systems save data. For a single small file you can convert to pandas first.

In [ ]:
# Spark-native write (creates a folder of part-files)
by_class.write.csv(DATA_PATH + "survival_by_class_spark", header=True, mode="overwrite")

# Or collect a small result to pandas for a single CSV
by_class.toPandas().to_csv("/dbfs/FileStore/cbs_datasets/Session_5/survival_by_class.csv", index=False)

<a id="takeaways"></a>
## Key Takeaways

| Concept / command | What it does |
|---|---|
| `SparkSession.builder...getOrCreate()` | Start (or reuse) the entry point to Spark |
| Transformations vs actions | Transformations are lazy (`select`, `filter`); actions run the work (`show`, `count`) |
| `StructType` schema | Name and type the columns (essential for headerless files) |
| `spark.read.csv(...)` | Read files into a distributed DataFrame |
| `select`, `withColumn`, `withColumnRenamed` | Choose, add and rename columns (returns a new DataFrame) |
| `filter` | Keep rows, via a SQL string or column expressions (`&`, `|`, `~`) |
| `.na.drop` / `.na.fill` | Handle missing values |
| `groupBy(...).agg(...)` | Grouped aggregations |
| `toPandas()` | Bring a small/aggregated result into pandas to plot; never on huge data |
| `collect()` caution | Never pull a huge DataFrame to the driver |

**The one-line lesson:** Spark DataFrames give you pandas-like operations that scale to data too big for one machine; the new ideas are **lazy evaluation** and the **aggregate-in-Spark, plot-in-pandas** pattern via `toPandas()`.

## Conclusion

You can now start a Spark session, define schemas, load and inspect distributed DataFrames, transform and filter them, handle missing data, aggregate, plot via `toPandas()`, and query with SQL, all on one realistic dataset. The next notebook builds a machine-learning model on top of these skills with **Spark MLlib**.

<a id="reading"></a>
## Further Reading & Resources

- [PySpark DataFrame quickstart](https://spark.apache.org/docs/latest/api/python/getting_started/quickstart_df.html)
- [Spark SQL and DataFrames](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [Installing PySpark / running on Colab](https://spark.apache.org/docs/latest/api/python/getting_started/install.html)